<a href="https://colab.research.google.com/github/pablobelmiro/olist_exploration/blob/main/03_features_e_selecao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce: Capítulo 3, Features e Seleção por Cenário

Continuação dos capítulos 1 e 2. Esse notebook parte do zero de novo (carrega os 9 CSVs e remonta o dataframe mestre), autocontido, mas assume que você já leu os dois anteriores.

Precisa de `scikit-learn` além do pandas de sempre (`pip install scikit-learn` se não tiver). Não precisa de T4, os modelos aqui são só diagnóstico de feature, rápidos de treinar em CPU.

In [2]:
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.feature_selection import mutual_info_classif

orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders[col] = pd.to_datetime(orders[col])

produtos_com_categoria_en = products.merge(category_translation, on='product_category_name', how='left')

mestre = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(produtos_com_categoria_en, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments, on='order_id', how='left')
    .merge(reviews, on='order_id', how='left')
)

print(f"dataframe mestre: {mestre.shape}")

dataframe mestre: (118310, 40)


## Feature nova: distância cliente-vendedor

O capítulo 2 já mostrou que estado do cliente correlaciona com tempo de entrega. Aqui eu calculo uma distância de verdade (Haversine, em km) entre cliente e vendedor, usando a latitude/longitude média por prefixo de CEP do `olist_geolocation_dataset`. Essa tabela tem mais de 1 milhão de linhas (várias coordenadas por prefixo), por isso a média por prefixo primeiro.

In [3]:
geo_media = geolocation.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

def haversine(lat1, lon1, lat2, lon2):
    """Distância em km entre dois pontos lat/lng, aproximando a Terra como esfera de raio 6371km."""
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

m = mestre.merge(
    geo_media.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix', 'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'}),
    on='customer_zip_code_prefix', how='left',
)
m = m.merge(
    geo_media.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix', 'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'}),
    on='seller_zip_code_prefix', how='left',
)
m['distance_km'] = haversine(m['cust_lat'], m['cust_lng'], m['sell_lat'], m['sell_lng'])

print(m['distance_km'].describe())
print(f"\nnulos (prefixo de CEP sem coordenada na base de geolocalização): {m['distance_km'].isnull().sum()} / {len(m)}")

count    35022.000000
mean       141.297832
std        137.057895
min          0.000000
25%         26.769831
50%         87.677294
75%        249.600417
max        669.957376
Name: distance_km, dtype: float64

nulos (prefixo de CEP sem coordenada na base de geolocalização): 83288 / 118310


## Cenário 1: Atraso na entrega

Candidatas: `distance_km`, `product_weight_g`, `price`, `freight_value`, tempo de aprovação (`approval_hours`), mês da compra (sazonalidade). Uma linha por pedido aqui (não por item), pra não enviesar o modelo com pedidos de muitos itens.

**Cuidado com vazamento de dado**: o alvo `atrasado` é definido comparando `order_delivered_customer_date` com `order_estimated_delivery_date`. Se eu incluir o próprio `atraso_dias` (a diferença calculada) como feature, o modelo não está aprendendo a prever atraso, está literalmente lendo a resposta. Vou treinar dois modelos, um honesto (sem `atraso_dias`) e um vazado (com), pra mostrar a diferença na prática.

In [4]:
m['approval_hours'] = (m['order_approved_at'] - m['order_purchase_timestamp']).dt.total_seconds() / 3600
m['month'] = m['order_purchase_timestamp'].dt.month
m['atraso_dias'] = (m['order_delivered_customer_date'] - m['order_estimated_delivery_date']).dt.days

delay_df = (
    m.drop_duplicates(subset='order_id')
    [['order_id', 'distance_km', 'product_weight_g', 'price', 'freight_value', 'approval_hours', 'month', 'atraso_dias']]
    .dropna()
)
delay_df['atrasado'] = (delay_df['atraso_dias'] > 0).astype(int)

print(f"pedidos usados: {len(delay_df)}")
print(f"taxa de atraso: {delay_df['atrasado'].mean():.4f} ({delay_df['atrasado'].sum()} de {len(delay_df)})")

pedidos usados: 28325
taxa de atraso: 0.0471 (1333 de 28325)


In [5]:
features_honestas = ['distance_km', 'product_weight_g', 'price', 'freight_value', 'approval_hours', 'month']
features_vazadas = features_honestas + ['atraso_dias']

X = delay_df[features_honestas]
y = delay_df['atrasado']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

modelo_honesto = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
modelo_honesto.fit(Xtr, ytr)
acc_honesto = accuracy_score(yte, modelo_honesto.predict(Xte))
auc_honesto = roc_auc_score(yte, modelo_honesto.predict_proba(Xte)[:, 1])
print(f"HONESTO (sem atraso_dias): acurácia {acc_honesto:.4f}, AUC {auc_honesto:.4f}")
print(f"(taxa de atraso é {y.mean():.4f}, então prever sempre 'no prazo' já dá acurácia ~{1-y.mean():.4f}, acurácia sozinha engana aqui, AUC é a métrica que importa)")

Xl = delay_df[features_vazadas]
Xltr, Xlte, yltr, ylte = train_test_split(Xl, y, test_size=0.2, random_state=42, stratify=y)
modelo_vazado = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
modelo_vazado.fit(Xltr, yltr)
acc_vazado = accuracy_score(ylte, modelo_vazado.predict(Xlte))
auc_vazado = roc_auc_score(ylte, modelo_vazado.predict_proba(Xlte)[:, 1])
print(f"\nVAZADO (com atraso_dias): acurácia {acc_vazado:.4f}, AUC {auc_vazado:.4f}")

HONESTO (sem atraso_dias): acurácia 0.9529, AUC 0.6775
(taxa de atraso é 0.0471, então prever sempre 'no prazo' já dá acurácia ~0.9529, acurácia sozinha engana aqui, AUC é a métrica que importa)

VAZADO (com atraso_dias): acurácia 1.0000, AUC 1.0000


In [6]:
leakage_json = [
    {'model': 'honesto (sem atraso_dias)', 'auc': round(float(auc_honesto), 4)},
    {'model': 'vazado (com atraso_dias)', 'auc': round(float(auc_vazado), 4)},
]
with open('leakage-comparison.json', 'w', encoding='utf-8') as f:
    json.dump(leakage_json, f, ensure_ascii=False, indent=2)
print(leakage_json)

importancias = sorted(zip(features_honestas, modelo_honesto.feature_importances_), key=lambda x: -x[1])
print("\nimportância das features (modelo honesto):")
for feat, imp in importancias:
    print(f"  {feat}: {imp:.4f}")

importance_json = [{'feature': feat, 'importance': round(float(imp), 4)} for feat, imp in importancias]
with open('delay-feature-importance.json', 'w', encoding='utf-8') as f:
    json.dump(importance_json, f, ensure_ascii=False, indent=2)

[{'model': 'honesto (sem atraso_dias)', 'auc': 0.6775}, {'model': 'vazado (com atraso_dias)', 'auc': 1.0}]

importância das features (modelo honesto):
  approval_hours: 0.2124
  price: 0.1728
  freight_value: 0.1701
  distance_km: 0.1680
  product_weight_g: 0.1580
  month: 0.1187


## Cenário 2: Nota da avaliação

Candidatas: `atraso_dias`, tempo de entrega, preço, frete, número de parcelas. Uso `mutual_info_classif` (informação mútua) em vez de correlação simples, porque `review_score` é categórico ordinal (1 a 5), não uma escala contínua bem comportada (o capítulo 2 já mostrou isso, é bimodal).

In [7]:
m['delivery_days'] = (m['order_delivered_customer_date'] - m['order_purchase_timestamp']).dt.days

s2 = (
    m.drop_duplicates(subset='order_id')
    [['order_id', 'atraso_dias', 'delivery_days', 'price', 'freight_value', 'payment_installments', 'review_score']]
    .dropna()
)

X2 = s2[['atraso_dias', 'delivery_days', 'price', 'freight_value', 'payment_installments']]
y2 = s2['review_score']
mi = mutual_info_classif(X2, y2, random_state=42)

print(f"pedidos usados: {len(s2)}\n")
print("informação mútua com review_score:")
for feat, val in sorted(zip(X2.columns, mi), key=lambda x: -x[1]):
    print(f"  {feat}: {val:.4f}")

pedidos usados: 95829

informação mútua com review_score:
  atraso_dias: 0.0684
  delivery_days: 0.0571
  freight_value: 0.0077
  price: 0.0074
  payment_installments: 0.0026


## Cenário 3: Valor do frete

Candidatas: peso e as três dimensões do produto. Correlação de Pearson simples aqui, é suficiente pra ver qual domina.

In [8]:
s3 = (
    m.drop_duplicates(subset='order_id')
    [['order_id', 'freight_value', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']]
    .dropna()
)

print(f"pedidos usados: {len(s3)}\n")
correlacao_frete = s3[['freight_value', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].corr()['freight_value'].sort_values(ascending=False)
print(correlacao_frete)

pedidos usados: 98650

freight_value        1.000000
product_weight_g     0.615310
product_height_cm    0.393193
product_width_cm     0.330805
product_length_cm    0.317080
Name: freight_value, dtype: float64


## Cenário 4: Segmentação de clientes (RFM)

Recência (dias desde a última compra), Frequência (número de pedidos), Valor monetário (soma paga). Isso não é seleção de feature no sentido supervisionado, é a engenharia das 3 features que vão alimentar o clustering do capítulo 5. Padronização (StandardScaler) fica pra lá também, junto do KMeans de verdade.

In [9]:
data_max = orders['order_purchase_timestamp'].max()
print(f"data mais recente no dataset: {data_max}")

orders_validos = orders[orders['order_status'] != 'canceled']
oc = orders_validos.merge(customers, on='customer_id', how='left')
pagamento_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()
oc = oc.merge(pagamento_por_pedido, on='order_id', how='left')

rfm = oc.groupby('customer_unique_id').agg(
    recencia_dias=('order_purchase_timestamp', lambda x: (data_max - x.max()).days),
    frequencia=('order_id', 'nunique'),
    valor_monetario=('payment_value', 'sum'),
).reset_index()

print(f"\nclientes únicos: {len(rfm)}")
print(rfm[['recencia_dias', 'frequencia', 'valor_monetario']].describe())

recorrentes = (rfm['frequencia'] > 1).sum()
print(f"\nclientes com mais de 1 pedido: {recorrentes} ({100*recorrentes/len(rfm):.2f}% do total)")

data mais recente no dataset: 2018-10-17 17:30:18

clientes únicos: 95560
       recencia_dias    frequencia  valor_monetario
count   95560.000000  95560.000000     95560.000000
mean      287.562422      1.034073       166.027799
std       153.123799      0.212154       227.761687
min        44.000000      1.000000         0.000000
25%       163.000000      1.000000        63.110000
50%       268.000000      1.000000       107.940000
75%       397.000000      1.000000       183.180000
max       772.000000     17.000000     13664.080000

clientes com mais de 1 pedido: 2924 (3.06% do total)
